# NeuroGolf 2026 Score Plateau Triage

This notebook investigates why a new solver revision can validate more tasks locally while the Kaggle public score remains unchanged. It compares solver manifests across runs, highlights newly covered tasks, renders task panels for review, and writes lightweight triage artifacts.

Use this after running `5_simple_solver_export.ipynb`. Attach one or more Kaggle output datasets that contain `simple_logic_manifest.csv` files, or run it in the same session where the manifest is available under `/kaggle/working`.

# 1. Setup and Configuration
## 1.1 Paths and Runtime Flags

The notebook is analysis-only. It never creates `submission.zip`; it reads manifests and task JSON files, then writes CSV/figure artifacts that explain solver coverage.

In [1]:
from __future__ import annotations

INPUT_ROOT = "/kaggle/input"
WORKING_ROOT = "/kaggle/working"
TASK_DIR_CANDIDATES = (
    "/kaggle/input/competitions/neurogolf-2026",
    "/kaggle/input/neurogolf-2026",
)
MANIFEST_NAME = "simple_logic_manifest.csv"
MAX_RENDER_TASKS = 12
KNOWN_PUBLIC_SCORE = 253.94
PRIOR_SOLVED_TASKS = 60
CURRENT_NOTE = "Run after notebook 5 or attach its Kaggle output dataset."


### Configuration Notes

- `PRIOR_SOLVED_TASKS = 60` is the score-compatible baseline that repeatedly produced `253.94`.
- The notebook can compare multiple manifests when prior Kaggle output datasets are attached.
- When only one manifest is available, the notebook still reports solver mix, dynamic solver selections, and likely plateau reasons.

## 1.2 Imports and Output Paths

In [2]:
import json
import re
import zipfile
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path(INPUT_ROOT)
WORKING_ROOT = Path(WORKING_ROOT)
FIGURE_DIR = WORKING_ROOT / "score_triage_figures"
SUMMARY_PATH = WORKING_ROOT / "score_triage_solver_summary.csv"
DELTA_PATH = WORKING_ROOT / "score_triage_manifest_delta.csv"
DYNAMIC_PATH = WORKING_ROOT / "score_triage_dynamic_tasks.csv"
ZIP_PATH = WORKING_ROOT / "score_triage_artifacts.zip"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"input root exists: {INPUT_ROOT.exists()}")
print(f"working root exists: {WORKING_ROOT.exists()}")
print(CURRENT_NOTE)


input root exists: True
working root exists: True
Run after notebook 5 or attach its Kaggle output dataset.


# 2. Manifest Discovery
## 2.1 Find and Load Run Manifests

Each `simple_logic_manifest.csv` captures the task ids and solver families exported by one notebook-5 run. Comparing these files tells us whether a new notebook version actually selected new task models.

In [3]:
def discover_manifest_paths() -> list[Path]:
    """Find available simple-logic manifest files.

    Returns:
        Sorted manifest paths from Kaggle inputs and working output.
    """
    candidates = []
    search_roots = [INPUT_ROOT, WORKING_ROOT]
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob(MANIFEST_NAME))
    return sorted(set(candidates), key=lambda path: str(path))


def run_label(path: Path) -> str:
    """Create a compact run label from a manifest path.

    Args:
        path: Manifest path.

    Returns:
        Human-readable run label.
    """
    if path.parent == WORKING_ROOT:
        return "working_current"
    parts = [part for part in path.parts if part not in {"/", "kaggle", "input"}]
    label = "_".join(parts[-3:]) if len(parts) >= 3 else path.stem
    label = re.sub(r"[^A-Za-z0-9_]+", "_", label)
    return label.strip("_") or path.stem


def load_manifest(path: Path) -> pd.DataFrame:
    """Load one solver manifest with a run label.

    Args:
        path: Manifest CSV path.

    Returns:
        Manifest dataframe with source metadata columns.
    """
    df = pd.read_csv(path)
    df["run_label"] = run_label(path)
    df["manifest_path"] = str(path)
    return df


manifest_paths = discover_manifest_paths()
print(f"Found {len(manifest_paths)} manifest file(s)")
for path in manifest_paths:
    print(path)

manifests = [load_manifest(path) for path in manifest_paths]
manifest_df = pd.concat(manifests, ignore_index=True) if manifests else pd.DataFrame()
display(manifest_df.head(MAX_RENDER_TASKS))


Found 0 manifest file(s)


""


### Manifest Discovery Insights

- If no manifest is found, run notebook 5 first or attach Kaggle output datasets from previous notebook-5 versions.
- If one manifest is found, inspect solver mix and dynamic solver selections.
- If two or more manifests are found, compare task-level deltas to see whether the latest code selected any new task ids.

## 2.2 Solver Mix by Run

In [4]:
if manifest_df.empty:
    summary_df = pd.DataFrame()
else:
    summary_df = (
        manifest_df.groupby(["run_label", "solver_kind", "validation_scope"])
        .size()
        .rename("task_count")
        .reset_index()
        .sort_values(["run_label", "task_count"], ascending=[True, False])
    )
    display(summary_df)
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(SUMMARY_PATH, index=False)
    print(f"Wrote {SUMMARY_PATH}")


### Solver Mix Insights

The first question is whether new solver families are actually being selected. If `dynamic_anchor_crop` or `dynamic_bbox_crop` is absent, then the unchanged score is expected: the new code did not affect the submitted task set.

# 3. Run Comparison
## 3.1 Select Baseline and Candidate Runs

When multiple manifests are attached, the notebook compares the smallest solved-task run as the baseline against the largest solved-task run as the candidate. This is usually the useful comparison for score-plateau debugging.

In [5]:
def select_runs(df: pd.DataFrame) -> tuple[str | None, str | None]:
    """Select baseline and candidate run labels.

    Args:
        df: Combined manifest dataframe.

    Returns:
        Baseline and candidate labels, or `(None, None)`.
    """
    if df.empty:
        return None, None
    counts = df.groupby("run_label")["task_id"].nunique().sort_values()
    if len(counts) == 1:
        label = counts.index[0]
        return label, label
    return counts.index[0], counts.index[-1]


baseline_label, candidate_label = select_runs(manifest_df)
print(f"baseline run: {baseline_label}")
print(f"candidate run: {candidate_label}")


baseline run: None
candidate run: None


## 3.2 Task-Level Delta

In [6]:
def safe_value(df: pd.DataFrame, column: str) -> Any:
    """Return the first value for a column when available.

    Args:
        df: Dataframe slice.
        column: Column to read.

    Returns:
        First column value, or None.
    """
    if df.empty or column not in df:
        return None
    return df[column].iloc[0]


def manifest_delta(
    df: pd.DataFrame, baseline_label: str, candidate_label: str
) -> pd.DataFrame:
    """Compare task coverage between two manifest runs.

    Args:
        df: Combined manifest dataframe.
        baseline_label: Baseline run label.
        candidate_label: Candidate run label.

    Returns:
        Task-level delta dataframe.
    """
    base = df[df["run_label"] == baseline_label].copy()
    cand = df[df["run_label"] == candidate_label].copy()
    base_tasks = set(base["task_id"])
    cand_tasks = set(cand["task_id"])
    rows = []
    for task_id in sorted(base_tasks | cand_tasks):
        base_row = base[base["task_id"] == task_id]
        cand_row = cand[cand["task_id"] == task_id]
        if task_id in base_tasks and task_id in cand_tasks:
            status = "kept"
        elif task_id in cand_tasks:
            status = "added"
        else:
            status = "removed"
        rows.append(
            {
                "task_id": task_id,
                "status": status,
                "baseline_solver": safe_value(base_row, "solver_kind"),
                "candidate_solver": safe_value(cand_row, "solver_kind"),
                "baseline_cost": safe_value(base_row, "cost_estimate"),
                "candidate_cost": safe_value(cand_row, "cost_estimate"),
            }
        )
    return pd.DataFrame(rows)


if baseline_label is None or candidate_label is None:
    delta_df = pd.DataFrame()
elif baseline_label == candidate_label:
    delta_df = manifest_df[["task_id", "solver_kind", "run_label"]].copy()
    delta_df["status"] = "single_run"
else:
    delta_df = manifest_delta(manifest_df, baseline_label, candidate_label)

display(delta_df.head(50))
if not delta_df.empty:
    delta_df.to_csv(DELTA_PATH, index=False)
    print(f"Wrote {DELTA_PATH}")
    display(delta_df["status"].value_counts().rename("task_count"))


""


### Delta Insights

Interpret the plateau this way:

- `added = 0`: the new notebook version did not change submitted task coverage.
- `added > 0` and score unchanged: new tasks are probably not in the public scored slice, or their inferred rule does not generalize to the scored test inputs.
- many `removed` tasks: candidate selection or cost ranking may have regressed and should be inspected before another submission.

# 4. Dynamic Solver Focus
## 4.1 Identify Anchor and Bounding-Box Crop Selections

This section isolates the exact task ids affected by the new object-crop logic.

In [7]:
if manifest_df.empty or "solver_kind" not in manifest_df:
    dynamic_df = pd.DataFrame()
else:
    dynamic_df = manifest_df[
        manifest_df["solver_kind"].astype(str).str.contains(
            "dynamic|bbox|anchor", case=False, regex=True
        )
    ].copy()
    dynamic_df = dynamic_df.sort_values(["run_label", "task_id"])
    display(dynamic_df)
    dynamic_df.to_csv(DYNAMIC_PATH, index=False)
    print(f"Wrote {DYNAMIC_PATH}")


### Dynamic Solver Insights

If this table is empty, the dynamic solver code is not yet matching train/public examples. The next work should improve candidate detection before adding more ONNX builders. If this table has rows but the score is unchanged, render those tasks and inspect whether the rule is too specific to the available examples.

# 5. Task Rendering
## 5.1 Load Task JSON Files

In [8]:
def find_task_dir() -> Path | None:
    """Find the competition task directory.

    Returns:
        Directory containing `task*.json`, or None.
    """
    for candidate in TASK_DIR_CANDIDATES:
        path = Path(candidate)
        if path.exists() and list(path.glob("task*.json")):
            return path
    if INPUT_ROOT.exists():
        for path in INPUT_ROOT.rglob("task001.json"):
            return path.parent
    return None


def load_tasks(task_dir: Path | None) -> dict[str, dict[str, Any]]:
    """Load NeuroGolf task JSON files.

    Args:
        task_dir: Directory containing task JSON files.

    Returns:
        Mapping from task id to payload.
    """
    if task_dir is None:
        return {}
    tasks = {}
    for path in sorted(task_dir.glob("task*.json")):
        tasks[path.stem] = json.loads(path.read_text())
    return tasks


task_dir = find_task_dir()
tasks = load_tasks(task_dir)
print(f"task dir: {task_dir}")
print(f"loaded tasks: {len(tasks)}")


task dir: /kaggle/input/competitions/neurogolf-2026
loaded tasks: 400


## 5.2 Render Added or Dynamic Tasks

In [9]:
ARC_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25",
]
ARC_CMAP = plt.matplotlib.colors.ListedColormap(ARC_COLORS)


def render_grid(ax, grid: list[list[int]], title: str) -> None:
    """Render one ARC grid.

    Args:
        ax: Matplotlib axis.
        grid: Integer color grid.
        title: Axis title.
    """
    arr = np.asarray(grid)
    ax.imshow(arr, cmap=ARC_CMAP, vmin=0, vmax=9, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#222222", linewidth=0.25)


def render_task(task_id: str, task: dict[str, Any], note: str) -> Path:
    """Render train and public test examples for one task.

    Args:
        task_id: Task id.
        task: Task payload.
        note: Plot subtitle note.

    Returns:
        Figure path.
    """
    pairs = []
    for split_name in ("train", "test"):
        for index, pair in enumerate(task.get(split_name, []), start=1):
            if "input" in pair and "output" in pair:
                pairs.append((split_name, index, pair))
    if not pairs:
        raise ValueError(f"No renderable pairs for {task_id}")
    fig, axes = plt.subplots(
        len(pairs), 2, figsize=(6, max(2.4, 2.2 * len(pairs)))
    )
    axes = np.asarray(axes).reshape(len(pairs), 2)
    for row, (split_name, index, pair) in enumerate(pairs):
        render_grid(axes[row, 0], pair["input"], f"{split_name} {index} input")
        render_grid(axes[row, 1], pair["output"], f"{split_name} {index} output")
    fig.suptitle(f"{task_id}: {note}", fontsize=11)
    fig.tight_layout()
    safe_note = re.sub(r"[^A-Za-z0-9]+", "_", note).strip("_")
    path = FIGURE_DIR / f"{task_id}_{safe_note}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


if tasks:
    if not dynamic_df.empty:
        render_ids = dynamic_df["task_id"].drop_duplicates().head(MAX_RENDER_TASKS)
        note_lookup = dynamic_df.drop_duplicates("task_id").set_index("task_id")[
            "solver_kind"
        ].to_dict()
    elif not delta_df.empty and "status" in delta_df:
        added = delta_df[delta_df["status"] == "added"]
        render_ids = added["task_id"].drop_duplicates().head(MAX_RENDER_TASKS)
        note_lookup = added.set_index("task_id")["candidate_solver"].to_dict()
    else:
        render_ids = []
        note_lookup = {}
    figure_rows = []
    for task_id in render_ids:
        if task_id not in tasks:
            continue
        note = str(note_lookup.get(task_id, "triage"))
        figure_path = render_task(task_id, tasks[task_id], note)
        figure_rows.append(
            {"task_id": task_id, "note": note, "figure_path": str(figure_path)}
        )
    figure_df = pd.DataFrame(figure_rows)
    display(figure_df)
else:
    figure_df = pd.DataFrame()
    print("Task files were not found, so rendering was skipped.")


""


### Rendering Insights

Rendered panels are the fastest way to decide whether the new solver family is plausible. For dynamic crop tasks, inspect whether the selected output is truly anchored by a unique marker or whether the train examples accidentally fit a narrower rule.

# 6. Plateau Diagnosis and Artifacts
## 6.1 Generate a Short Diagnosis

In [10]:
def build_diagnosis(
    manifest_df: pd.DataFrame, delta_df: pd.DataFrame, dynamic_df: pd.DataFrame
) -> list[str]:
    """Build human-readable plateau diagnosis bullets.

    Args:
        manifest_df: Combined manifest dataframe.
        delta_df: Task-level delta dataframe.
        dynamic_df: Dynamic solver dataframe.

    Returns:
        Diagnosis bullet strings.
    """
    bullets = []
    if manifest_df.empty:
        return [
            "No manifest found. Run notebook 5 or attach Kaggle output datasets."
        ]
    run_counts = manifest_df.groupby("run_label")["task_id"].nunique()
    latest_count = int(run_counts.max())
    bullets.append(f"Largest attached run solves {latest_count} task models.")
    if latest_count <= PRIOR_SOLVED_TASKS:
        bullets.append(
            "Coverage did not exceed the repeated 60-task public-score baseline."
        )
    else:
        bullets.append(
            "Coverage exceeds the repeated 60-task baseline, but public score "
            "must confirm whether those tasks affect the leaderboard slice."
        )
    if not delta_df.empty and "status" in delta_df:
        added_count = int((delta_df["status"] == "added").sum())
        bullets.append(f"Manifest comparison shows {added_count} added task ids.")
        if added_count and KNOWN_PUBLIC_SCORE == 253.94:
            bullets.append(
                "If public score is still 253.94, the added tasks likely do not "
                "overlap the scored public cases or do not generalize to scorer inputs."
            )
    if dynamic_df.empty:
        bullets.append(
            "No dynamic crop solver rows were selected in the attached manifests."
        )
    else:
        bullets.append(
            f"Dynamic crop solvers selected {dynamic_df['task_id'].nunique()} task ids."
        )
    return bullets


diagnosis = build_diagnosis(manifest_df, delta_df, dynamic_df)
for item in diagnosis:
    print(f"- {item}")


- No manifest found. Run notebook 5 or attach Kaggle output datasets.


## 6.2 Zip Triage Artifacts

In [11]:
artifact_paths = [SUMMARY_PATH, DELTA_PATH, DYNAMIC_PATH]
if "figure_df" in globals() and not figure_df.empty:
    artifact_paths.extend(Path(path) for path in figure_df["figure_path"])

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        path = Path(path)
        if path.exists():
            zf.write(path, arcname=path.name)

print(f"Wrote {ZIP_PATH}")


Wrote /kaggle/working/score_triage_artifacts.zip


### Final Triage Notes

Use the artifact zip to bring the manifest summaries and task panels back into the repo discussion. The next modeling decision should be based on whether dynamic crop solvers were selected and whether any newly selected task ids are visually convincing.